# Folk-painting restoration (CS F407) - Kaggle runner
**Before you run anything, in the right-hand *Settings* panel:**
1. **Accelerator -> GPU** (T4 x2 or P100; the code uses one GPU)
2. **Internet -> On** (needed for dataset download AND pretrained weights: ResNet-34, LaMa, LPIPS). Kaggle asks for phone verification once.
3. **Add Input -> Upload / your datasets -> add the dataset that holds the code** (`folk-restore.zip` uploaded as a dataset, e.g. named `folk-restore-code`).

Everything below runs inside `/kaggle/working/folk-restore` (that folder is your notebook's *Output*). `/kaggle/input` is read-only.

**v2:** sections E1-E6 (E1b/E1c = v3 wall-damage segmenter and the threshold/scale diagnosis) (after the original steps) add the segmenter fix, the fine-tuned LaMa (M2), the Pix2Pix de-fader (M3), Real-ESRGAN + uncertainty map (M4), the one-command `restore.py` and the validation notebook.

In [ ]:
# ---- 1. put the code in /kaggle/working/folk-restore and cd into it
import os, glob, shutil, zipfile, hashlib
WORK = "/kaggle/working/folk-restore"
REFRESH = True   # True = re-copy the code from /kaggle/input every time you run this cell (your data/ results are never touched)

def find_code(base="/kaggle/input", max_depth=7):
    """Find the folder that holds common.py + degrade.py, however deep Kaggle mounted it."""
    for root, dirs, files in os.walk(base):
        if "common.py" in files and "degrade.py" in files:
            return root
        if root[len(base):].count(os.sep) >= max_depth:
            dirs[:] = []
    return None

if REFRESH or not os.path.exists(f"{WORK}/common.py"):
    src = find_code()
    if src is None:  # Kaggle may have kept the upload as an un-extracted .zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith("common.py") for n in zf.namelist()):
                    shutil.rmtree("/kaggle/working/_code", ignore_errors=True)
                    zf.extractall("/kaggle/working/_code")
                    break
        src = find_code("/kaggle/working/_code")
    if src is None:
        print("Nothing under /kaggle/input looks like the code. What is attached to this notebook:")
        os.system("find /kaggle/input -maxdepth 4 | head -40")
        raise FileNotFoundError("Code not found. Open the 'Input' panel on the right, click 'Add Input' and attach the dataset that contains folk-restore.zip.")
    shutil.copytree(src, WORK, dirs_exist_ok=True)
    print("code copied from:", src)
os.chdir(WORK)
print("working dir:", os.getcwd())

In [ ]:
# ---- 2. install the few missing packages, check the GPU
!pip install -q segmentation-models-pytorch timm lpips requests
!pip install -q simple-lama-inpainting --no-deps
import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## A. Collect the dataset
Target: **~60-100 images per style, short side >= 800 px**, open licences. Run the automatic collector first, then top up with Kaggle datasets or your own downloads.

In [ ]:
# (optional) discover the right Wikimedia Commons category names, then paste them into 00_collect_dataset.py
!python 00_collect_dataset.py find-categories --term Madhubani

In [ ]:
# ---- automatic download: Wikimedia Commons + The Met + Cleveland Museum of Art + Art Institute of Chicago
# ~10-25 min. Safe to re-run (it skips files it already has). Use your real email: Wikimedia asks for a contact.
CONTACT = "bhalerao.malhar@gmail.com"
!python 00_collect_dataset.py collect --contact {CONTACT} --sources commons --per_source 70 --max_per_style 150

In [ ]:
# ---- (optional) add images from Kaggle datasets you attached via "Add Input" (search: Madhubani, Warli, Pattachitra, Kalamkari, Indian painting)
# Check each dataset page's licence. Replace the paths/names below, then un-comment.
# !python 00b_ingest_folder.py --src /kaggle/input/<dataset-folder> --style madhubani --source_name kaggle-<name> --license "see dataset page"
# !python 00b_ingest_folder.py --src /kaggle/input/<dataset-folder> --auto_style --source_name kaggle-<name>

In [ ]:
# ---- how many images per style do we have?
import glob, os
for s in ["madhubani", "warli", "pattachitra", "kalamkari", "unsorted"]:
    n = len([f for f in glob.glob(f"data/raw/{s}/*") if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"))])
    if n: print(f"{s:12s} {n}")
print("\nlicences / sources:  data/sources.csv")

In [ ]:
import csv, os, shutil
R = lambda a, b: list(range(a, b + 1))
KEEP = (R(2, 5) + R(7, 16) + [18, 19, 20] + R(22, 36) + [39, 40, 42, 45, 53, 54]            # kalamkari
        + R(65, 76) + [134, 138, 139, 140, 141, 142, 147, 155]                               # madhubani
        + [190, 191, 192, 194, 195, 198, 199, 200, 201, 202, 207] + R(211, 217)              # pattachitra
        + [220, 222, 223, 226, 227, 228, 230]
        + [245, 246, 247, 250, 254, 255, 257, 259, 260, 263, 265, 266, 267, 272, 274, 280, 283, 284, 285, 287]
        + R(288, 292) + [306, 308, 311, 314, 318, 320, 327, 329, 330] + R(332, 340)          # warli
        + R(342, 348) + [352, 353])
DAMAGED = [13, 14, 19, 42, 45, 54, 190, 191, 223, 226]   # candidates for the real-damaged demo set (look at them first!)

rows = list(csv.DictReader(open("sheets/index.csv")))
assert len(rows) == 356, f"index.csv has {len(rows)} rows, expected 356. Did you regenerate the sheets? The numbers would no longer match."
keep, damaged = set(KEEP), set(DAMAGED)
moved = {"kept in data/raw": 0, "-> data/real_damaged": 0, "-> data/junk": 0}
for r in rows:
    i, p = int(r["id"]), r["path"]
    if not os.path.exists(p):
        continue
    if i in damaged:
        dst, key = os.path.join("data/real_damaged", os.path.basename(p)), "-> data/real_damaged"
    elif i not in keep:
        dst, key = p.replace("data/raw", "data/junk", 1), "-> data/junk"
    else:
        moved["kept in data/raw"] += 1
        continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(p, dst)
    moved[key] += 1
print(moved)
for s in ["kalamkari", "madhubani", "pattachitra", "warli"]:
    print(f"{s:12s} left in data/raw: {len(os.listdir(f'data/raw/{s}'))}")

## B. Pick 5-10 genuinely damaged paintings for the demo (do this BEFORE step 1)
Browse the numbered contact sheets, note the numbers of paintings with real cracks / flaking / stains / fading, and move them out of `data/raw` so they are not used as "clean" training images. Museum items (`met_`, `cma_`, `aic_`) are older, so they are more likely damaged.

In [ ]:
PREFIX = ""          # e.g. "met_" to only look at Met images, "" for everything
!python 00c_contact_sheet.py --prefix "{PREFIX}"
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("sheets/sheet_*.jpg"))[:4]:      # change the slice to see more sheets
    print(f); display(Image(f))

In [ ]:
PICK = []            # <- the numbers you chose, e.g. [12, 40, 77, 81, 95]
ids = " ".join(map(str, PICK))
if PICK:
    !python 00c_contact_sheet.py --copy {ids}
print(len(glob.glob("data/real_damaged/*")), "images in data/real_damaged")

## C. Run the pipeline
`01` split by image + tile -> `02` fixed synthetic val/test sets -> smoke test -> `03` train -> `04` evaluate -> `06` de-fade baseline -> `05` full pipeline on real images -> `07` figures.

In [ ]:
!python 01_prepare_tiles.py --stride 256 --max_tiles 80 --min_side 500 --test_frac 0.15
!python 02_make_synthetic.py

In [ ]:
# 2-minute smoke test: if this fails, fix it before spending an hour on training
!python 03_train_unet.py --epochs 1 --samples_per_epoch 32 --bs 4 --workers 0 --n_val 8 --name smoke

In [ ]:
# ---- main training (~30-60 min). Keep this tab open.
!python 03_train_unet.py --name unet_rich --workers 3

In [ ]:
# ---- Milestone 1 + 2 numbers, and the Milestone 3 baseline
!python 04_evaluate.py --ckpt runs/unet_rich/best.pt
!python 06_defade_eval.py
print(open("results/summary.md").read())
print(open("results/seg_metrics.json").read())
print(open("results/defade_metrics.md").read())

In [ ]:
!python 08_train_defade.py --workers 3
print(open("results/defade_learned.md").read())

In [ ]:
# ---- full pipeline on the real damaged paintings (+ figures)
!python 05_restore_real.py --input_dir data/real_damaged --defade_model runs/defade/best.pt
!python 07_make_figures.py --dir results/samples --out figures/synthetic
!python 07_make_figures.py --dir results/real --out figures/real
!python 07_make_figures.py --dir results/defade_learned_samples --out figures/defade
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("figures/real/*.png"))[:4] + sorted(glob.glob("figures/synthetic/*.png"))[:3] + sorted(glob.glob("figures/defade/*.png"))[:2]:
    print(f); display(Image(f))

In [ ]:
# ---- OPTIONAL: CLAHE ablation (run while you write slides)
# !python 03_train_unet.py --mode rgb --name unet_rgb --workers 3
# !python 04_evaluate.py --ckpt runs/unet_rgb/best.pt --out results_rgb --no_lama
# print(open("results_rgb/seg_metrics.json").read())

In [ ]:
import os, cv2, numpy as np
import matplotlib.pyplot as plt

SRC = "results/real"        # change to e.g. "results/real_v2" to look at another run

def zoom_panels(stem, src=None, out="figures/zoom", n=3, half=110, min_area=150):
    """Crops around the n largest repaired regions: input | mask outline | LaMa result. Saves and returns the file path."""
    src = src or SRC
    rd = lambda f: cv2.cvtColor(cv2.imread(f"{src}/{stem}_{f}.png"), cv2.COLOR_BGR2RGB)
    inp, lama = rd("input"), rd("lama")
    mask = (cv2.imread(f"{src}/{stem}_mask.png", 0) > 0).astype(np.uint8)
    k, lab, st, cen = cv2.connectedComponentsWithStats(mask)
    comps = [i for i in np.argsort(-st[1:, cv2.CC_STAT_AREA]) + 1 if st[i, cv2.CC_STAT_AREA] >= min_area][:n]
    if not comps:
        print(f"{stem}: no repaired region >= {min_area}px, nothing to zoom into"); return None
    H, W = mask.shape
    fig, axes = plt.subplots(len(comps), 3, figsize=(10, 3.4 * len(comps)), squeeze=False)
    for r, i in enumerate(comps):
        cx, cy = map(int, cen[i])
        x0, y0 = max(0, min(cx - half, W - 2 * half)), max(0, min(cy - half, H - 2 * half))
        sl = (slice(y0, y0 + 2 * half), slice(x0, x0 + 2 * half))
        outline = inp[sl].copy()
        cnts, _ = cv2.findContours(mask[sl], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(outline, cnts, -1, (255, 255, 0), 1)
        for ax, im, t in zip(axes[r], (inp[sl], outline, lama[sl]), ("Input", "Detected damage", "LaMa fill")):
            ax.imshow(im); ax.set_title(t, fontsize=11); ax.axis("off")
    fig.tight_layout()
    os.makedirs(out, exist_ok=True)
    path = f"{out}/{stem}.png"; fig.savefig(path, dpi=130); plt.close(fig)
    return path

from IPython.display import Image, display
import glob
for stem in sorted({os.path.basename(f).rsplit("_", 1)[0] for f in glob.glob(f"{SRC}/*_mask.png")}):
    p = zoom_panels(stem, out=f"figures/zoom_{os.path.basename(SRC)}")
    if p: print(p); display(Image(p))

## E. Milestones 2-4, completed
**Before this section:** re-upload the new `folk-restore.zip` as a new version of your `folk-restore-code` dataset (cell 1 re-copies the code every run, `REFRESH = True`). Times are rough T4 estimates, not measurements. Run the cells in order. Each prints the numbers the final report needs.

### E1. Segmenter v2: design lines as hard negatives (fixes the Kalamkari failure)
Same U-Net, but 40% of training tiles get *unlabelled* dark vines, outlines and dotted borders, so the model learns that smooth uniform strokes are painting, not damage. ~15 min.

In [ ]:
SEG = "runs/unet_v2/best.pt"       # set to "runs/unet_rich/best.pt" to keep the old segmenter for everything below
!python 03_train_unet.py --name unet_v2 --decoy_prob 0.4 --workers 3
!python 04_evaluate.py --ckpt {SEG} --out results_v2 --no_lama
print(open("results_v2/seg_metrics.json").read())

### E1b. Segmenter v3: wall-mural damage, random scale, phone-photo look (targets the Mandawa failure)
The v2 simulator only imitated paper damage. v3 trains on a 50/50 mix of paper damage and **wall damage** (chalky plaster flaking that exposes plaster/ochre/brick, salt blooms, rain streaks), renders damage at a random 0.5-2x scale, and degrades 50% of inputs like a phone photo (blur, noise, JPEG, uneven light). It is checked on TWO fixed test sets: the original paper one and a new wall one. ~20 min.

In [ ]:
!python 02_make_synthetic.py --profile wall --scale_jitter 0.5 --photo 0.5 --out_name synthetic_wall --seed 321
!python 03_train_unet.py --name unet_v3 --decoy_prob 0.4 --profile mixed --scale_jitter 0.5 --photo 0.5 --val_dir data/synthetic_wall/val --workers 3
print("== v2 on the WALL test set ==")
!python 04_evaluate.py --ckpt runs/unet_v2/best.pt --test_dir data/synthetic_wall/test --out results_v2_wall --no_lama
print("== v3 on the WALL test set ==")
!python 04_evaluate.py --ckpt runs/unet_v3/best.pt --test_dir data/synthetic_wall/test --out results_v3_wall --no_lama
print("== v3 on the PAPER test set (must not get worse) ==")
!python 04_evaluate.py --ckpt runs/unet_v3/best.pt --out results_v3 --no_lama
for d in ("results_v2_wall", "results_v3_wall", "results_v3"):
    print(d, open(d + "/seg_metrics.json").read())
SEG = "runs/unet_v3/best.pt"       # from here on everything uses v3. If its paper-test Dice is clearly below v2's, set this back to runs/unet_v2/best.pt

### E1c. Why does it find so little on real photos? (threshold and scale sweep)
Prints, for each real image, the % of pixels flagged at three thresholds and five working scales, and saves panels (rows = scale, columns = image / thr 0.2 / 0.3 / 0.5). If a lower threshold or another scale shows sensible damage, the model is fine and only the operating point was wrong.

In [ ]:
!python 13_diagnose_real.py data/real_damaged --ckpt {SEG} --out results/diagnose
import glob, cv2, matplotlib.pyplot as plt
for f in sorted(glob.glob("results/diagnose/*_scales.png"))[:3]:
    plt.figure(figsize=(14, 12)); plt.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title(f); plt.show()

### E2. Milestone 2: fine-tune Big-LaMa on our tiles
Starts from the pretrained weights and trains on in-domain synthetic damage with L1 + perceptual + PatchGAN losses. Epoch 0 of the log is the zero-shot model, so the log shows directly whether fine-tuning helped. ~30-40 min.

In [ ]:
!python 09_finetune_lama.py --workers 3
!python 10_eval_inpaint.py --ckpt {SEG} --lama_ft runs/lama_ft/best.pt
print(open("results/inpaint_ft_summary.md").read())

### E3. Milestone 3: Pix2Pix de-fader with protected-margin references (+ an ablation without the GAN loss)
~15-25 min each.

In [ ]:
!python 11_train_pix2pix_defade.py --workers 3
print(open("results/defade_gan.md").read())
!python 11_train_pix2pix_defade.py --workers 3 --lambda_gan 0 --name defade_l1ref --out results_l1ref
print(open("results_l1ref/defade_gan.md").read())

### E4. Milestone 4: tiled Real-ESRGAN + uncertainty map, and whether the uncertainty is trustworthy
Downloads the official Real-ESRGAN x4 weights from GitHub (Internet must be On). Calibrates the discriminator on clean tiles, then checks the confidence map against the *true* restoration error on the synthetic test set. ~10 min.

In [ ]:
!python 12_eval_sr_uncertainty.py --ckpt {SEG} --lama_ft runs/lama_ft/best.pt --disc runs/lama_ft/disc.pt
print(open("results/uncertainty_eval.md").read())
print(open("results/sr_eval.md").read())

### E5. The integrated pipeline on the real paintings
One command: segment -> inpaint -> de-fade -> super-resolve -> confidence map. To use a protected margin as the colour reference add `--margin x0,y0,x1,y1` (pixels of an un-faded strip of the ORIGINAL image; one image at a time).

In [ ]:
!python restore.py data/real_damaged -o results/final --ckpt {SEG} --sr 2
import glob, cv2, matplotlib.pyplot as plt
for f in sorted(glob.glob("results/final/*_review.png"))[:4]:
    stem = f.replace("_review.png", "")
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a_, (s, t) in zip(ax, (("_restored.png", "restored"), ("_confidence.png", "confidence (red = speculative)"), ("_review.png", "review overlay"))):
        a_.imshow(cv2.cvtColor(cv2.imread(stem + s), cv2.COLOR_BGR2RGB)); a_.set_title(t); a_.axis("off")
    plt.show()

In [ ]:
# Real-image comparison panels (segmenter v1 vs v2 on the paintings where v1 erased design lines, e.g. the Kalamkari one)
!python 05_restore_real.py --input_dir data/real_damaged --ckpt runs/unet_rich/best.pt --out results/real_v1 --defade_model runs/defade/best.pt
!python 05_restore_real.py --input_dir data/real_damaged --ckpt {SEG} --out results/real_v2 --defade_model runs/defade/best.pt
!python 07_make_figures.py --dir results/real_v2 --out figures/real_v2

### E6. Validation notebook
Executes `validation.ipynb` (patch conversion, augmentation statistics, loss curves, all metrics) and saves the executed copy.

In [ ]:
!jupyter nbconvert --to notebook --execute validation.ipynb --output validation_executed.ipynb --ExecutePreprocessor.timeout=600

## D. Download your results
Run this cell, then download `deliverables.zip` and `raw_dataset.zip` from the **Output** panel (or `/kaggle/working`). Tip: do **not** press *Save Version* on a notebook with thousands of tile files; just download the zips.

In [ ]:
import zipfile, os
def pack(zip_path, items):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for it in items:
            if os.path.isdir(it):
                for root, _, files in os.walk(it):
                    for f in files: z.write(os.path.join(root, f))
            elif os.path.exists(it):
                z.write(it)
pack("/kaggle/working/deliverables.zip", ["results", "results_rgb", "results_v2", "results_v3", "results_v3_wall", "results_v2_wall", "results_l1ref", "figures", "sheets", "validation_executed.ipynb", "data/sources.csv",
    "runs/unet_rich/best.pt", "runs/unet_rich/log.csv", "runs/unet_v2/best.pt", "runs/unet_v2/log.csv", "runs/unet_v3/best.pt", "runs/unet_v3/log.csv", "runs/defade/best.pt", "runs/defade/log.csv",
    "runs/lama_ft/log.csv", "runs/lama_ft/disc_calib.json", "runs/defade_gan/best.pt", "runs/defade_gan/log.csv", "runs/defade_l1ref/log.csv"])
pack("/kaggle/working/raw_dataset.zip", ["data/raw", "data/real_damaged", "data/sources.csv"])   # re-upload as a Kaggle Dataset to skip collection next time
print({f: round(os.path.getsize(f) / 1e6, 1) for f in ["/kaggle/working/deliverables.zip", "/kaggle/working/raw_dataset.zip"]}, "MB")